In [13]:
import pandas as pd

# Load master agent table and raw transaction log
agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")
txns_raw_str = pd.read_csv("agent_transactions.csv", keep_default_na=False)

print("=== 1. AGENTS.CSV AUDIT ===")
print(f"Shape: {agents.shape[0]} rows x {agents.shape[1]} columns")
print("Missing values:\n", agents.isna().sum())
print(f"Duplicate rows: {agents.duplicated().sum()} | Unique agent_ids: {agents['agent_id'].nunique()}")
print(f"Onboarded window: {agents['onboarded'].min()} to {agents['onboarded'].max()}")
print("\nAgents per county:\n", agents["county"].value_counts())

print("\n=== 2. AGENT_TRANSACTIONS.CSV AUDIT ===")
print(f"Shape: {txns.shape[0]} rows x {txns.shape[1]} columns")
print("\nColumn types & missing values:")
print(pd.DataFrame({"dtype": txns.dtypes, "null_count": txns.isna().sum()}))

print("\nMissing-value breakdown in `amount`:")
print("  Empty cells (''):", (txns_raw_str["amount"] == "").sum())
print("  Literal 'N/A':   ", (txns_raw_str["amount"] == "N/A").sum())

print("\nNumeric summary of `amount` (raw):")
print(txns["amount"].describe().round(2))
print("  Injected 250,000 outliers:", (txns["amount"] == 250000).sum())

print("\nRaw `txn_type` categories:")
print(txns["txn_type"].value_counts())

print("\nDate format consistency (`txn_date`):")
print("  YYYY-MM-DD (hyphen):", txns["txn_date"].str.contains("-", regex=False).sum())
print("  YYYY/MM/DD (slash): ", txns["txn_date"].str.contains("/", regex=False).sum())
parsed_dates = pd.to_datetime(txns["txn_date"], format="mixed")
print(f"  Date range: {parsed_dates.min().date()} to {parsed_dates.max().date()}")

print("\nDuplicates & Referential Integrity:")
print("  Exact duplicate rows:  ", txns.duplicated().sum())
print("  Duplicate txn_id rows: ", txns.duplicated(subset=["txn_id"]).sum())
orphans = sorted(set(txns["agent_id"]) - set(agents["agent_id"]))
print("  Orphan agent_ids:      ", orphans)
print("  Rows with orphan ID:   ", (~txns["agent_id"].isin(agents["agent_id"])).sum())

=== 1. AGENTS.CSV AUDIT ===
Shape: 20 rows x 3 columns
Missing values:
 agent_id     0
county       0
onboarded    0
dtype: int64
Duplicate rows: 0 | Unique agent_ids: 20
Onboarded window: 2024-01-01 to 2025-08-01

Agents per county:
 county
Nairobi    6
Mombasa    5
Kisumu     4
Nakuru     3
Kiambu     2
Name: count, dtype: int64

=== 2. AGENT_TRANSACTIONS.CSV AUDIT ===
Shape: 612 rows x 5 columns

Column types & missing values:
            dtype  null_count
txn_id     object           0
agent_id   object           0
txn_type   object           0
amount    float64          22
txn_date   object           0

Missing-value breakdown in `amount`:
  Empty cells (''): 15
  Literal 'N/A':    7

Numeric summary of `amount` (raw):
count       590.00
mean       4537.20
std       22836.17
min         109.00
25%        1086.75
50%        1779.00
75%        3069.75
max      250000.00
Name: amount, dtype: float64
  Injected 250,000 outliers: 5

Raw `txn_type` categories:
txn_type
deposit       142


In [14]:
import pandas as pd

# Load master agent table and raw transaction log
agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

initial_rows = len(txns)

# 1. Inspect duplicate records (both exact row matches and duplicate txn_ids)
exact_dup_count = txns.duplicated(keep="first").sum()
id_dup_count = txns.duplicated(subset=["txn_id"], keep="first").sum()

print(f"Exact full-row duplicates: {exact_dup_count}")
print(f"Primary-key (txn_id) duplicates: {id_dup_count}")

# Preview any partial duplicates (same txn_id, differing values in other columns)
dup_id_rows = txns[txns.duplicated(subset=["txn_id"], keep=False)]
partial_dups = dup_id_rows[~dup_id_rows.duplicated(keep=False)].sort_values("txn_id")
print("\nPartial duplicate pairs (same txn_id, different column values):")
print(partial_dups.to_string(index=False))

# 2. Remove duplicates cleanly:
# Step A: Drop exact full-row duplicates
txns_dedup = txns.drop_duplicates(keep="first").copy()
exact_removed = initial_rows - len(txns_dedup)

# Step B: Drop remaining duplicate txn_ids, prioritizing rows with a valid agent_id
valid_agents = set(agents["agent_id"])
txns_dedup["_valid_agent"] = txns_dedup["agent_id"].isin(valid_agents)
txns_dedup = (
    txns_dedup.sort_values(by=["txn_id", "_valid_agent"], ascending=[True, False])
              .drop_duplicates(subset=["txn_id"], keep="first")
              .drop(columns=["_valid_agent"])
              .reset_index(drop=True)
)

key_only_removed = (initial_rows - exact_removed) - len(txns_dedup)
total_removed = initial_rows - len(txns_dedup)

print(f"\nRemoved {exact_removed} exact duplicates + {key_only_removed} key-only duplicate = {total_removed} total rows removed.")
print(f"Remaining unique transactions: {len(txns_dedup)}")

Exact full-row duplicates: 11
Primary-key (txn_id) duplicates: 12

Partial duplicate pairs (same txn_id, different column values):
 txn_id agent_id txn_type  amount   txn_date
TX00405    AG014  Deposit  1936.0 2026-04-03
TX00405    AG099  Deposit  1936.0 2026-04-03

Removed 11 exact duplicates + 1 key-only duplicate = 12 total rows removed.
Remaining unique transactions: 600


In [15]:
import numpy as np
import pandas as pd

# Load raw CSV (standard read_csv parses literal "N/A" as NaN automatically)
txns = pd.read_csv("agent_transactions.csv")

# Optional audit check with keep_default_na=False to see literal "N/A" strings before coercion
txns_literal = pd.read_csv("agent_transactions.csv", keep_default_na=False)

# Convert amount column to string before using .str accessor
print("Literal 'N/A' strings in amount:", (txns_literal["amount"].astype(str).str.strip().str.upper() == "N/A").sum())
print("Parsed NaN values in amount:    ", txns["amount"].isna().sum())

# 1. Check both isna() and invalid string sentinels on txn_type before normalizing
invalid_str_tokens = {"", "N/A", "NA", "NONE", "NULL", "NAN"}
missing_or_invalid_type = (
    txns["txn_type"].isna()
    | txns["txn_type"].astype(str).str.strip().str.upper().isin(invalid_str_tokens)
)
print(f"Missing or invalid txn_type rows: {missing_or_invalid_type.sum()}")

# 2. Trim whitespace and normalize casing -> "Deposit" or "Withdrawal"
txns["txn_type"] = np.where(
    missing_or_invalid_type,
    np.nan,
    txns["txn_type"].astype(str).str.strip().str.title(),
)

# 3. Enforce domain constraint: only "Deposit" and "Withdrawal" are permitted
allowed_types = {"Deposit", "Withdrawal"}
unexpected_values = set(txns["txn_type"].dropna()) - allowed_types
assert not unexpected_values, f"Unexpected txn_type values found: {unexpected_values}"

print("\nStandardized txn_type counts:")
print(txns["txn_type"].value_counts(dropna=False))

Literal 'N/A' strings in amount: 7
Parsed NaN values in amount:     22
Missing or invalid txn_type rows: 0

Standardized txn_type counts:
txn_type
Deposit       381
Withdrawal    231
Name: count, dtype: int64


In [16]:
import pandas as pd

# Load raw CSV (keep_default_na=False lets us inspect raw string anomalies explicitly)
txns = pd.read_csv("agent_transactions.csv", keep_default_na=False)

# --- 1. Convert `amount` to numeric (float64) ---
# Strip whitespace, optional currency prefixes ("KES"), and thousands commas before coercing
cleaned_amount_str = (
    txns["amount"]
    .astype(str)
    .str.strip()
    .str.replace(r"(?i)^kes\s*", "", regex=True)
    .str.replace(",", "", regex=False)
)
txns["amount"] = pd.to_numeric(cleaned_amount_str, errors="coerce")

# --- 2. Convert `txn_date` to datetime64[ns] ---
# Handles both 'YYYY-MM-DD' (607 rows) and 'YYYY/MM/DD' (5 rows) seamlessly
cleaned_date_str = txns["txn_date"].astype(str).str.strip()
txns["txn_date"] = pd.to_datetime(cleaned_date_str, format="mixed", errors="coerce")

# --- 3. Verify dtypes, null counts, and parsed ranges ---
print("Converted dtypes:")
print(txns.dtypes)

print("\nNull counts after type conversion:")
print(txns.isna().sum())

print(f"\nDate range: {txns['txn_date'].min().date()} to {txns['txn_date'].max().date()}")
print("\nNumeric summary of `amount` (590 valid numeric rows):")
print(txns["amount"].describe().round(2))

Converted dtypes:
txn_id              object
agent_id            object
txn_type            object
amount             float64
txn_date    datetime64[ns]
dtype: object

Null counts after type conversion:
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64

Date range: 2026-03-01 to 2026-04-29

Numeric summary of `amount` (590 valid numeric rows):
count       590.00
mean       4537.20
std       22836.17
min         109.00
25%        1086.75
50%        1779.00
75%        3069.75
max      250000.00
Name: amount, dtype: float64


In [17]:
import numpy as np
import pandas as pd

# Load raw CSV (keep_default_na=False preserves raw "" vs "N/A" strings for exact classification)
txns_raw = pd.read_csv("agent_transactions.csv", keep_default_na=False)
agents = pd.read_csv("agents.csv")

# Apply prior deduplication & type conversions
valid_agents = set(agents["agent_id"])
txns = txns_raw.copy()
txns["_valid_agent"] = txns["agent_id"].isin(valid_agents)
txns = (
    txns.drop_duplicates()
        .sort_values(["txn_id", "_valid_agent"], ascending=[True, False])
        .drop_duplicates(subset=["txn_id"], keep="first")
        .drop(columns=["_valid_agent"])
        .reset_index(drop=True)
)

raw_amount_str = txns["amount"].astype(str).str.strip()
txns["amount"] = pd.to_numeric(raw_amount_str, errors="coerce")
txns["txn_type"] = txns["txn_type"].astype(str).str.strip().str.title()
txns["txn_date"] = pd.to_datetime(txns["txn_date"].astype(str).str.strip(), format="mixed")

# Add completeness audit status & boolean flag column
conditions = [
    raw_amount_str == "",
    raw_amount_str.str.upper().isin(["N/A", "NA", "NONE", "NULL"]),
    txns["amount"].isna() | (txns["amount"] <= 0),
]
labels = ["Missing (Blank)", "Invalid ('N/A')", "Invalid (Non-positive)"]

txns["amount_status"] = np.select(conditions, labels, default="Valid")
txns["is_missing_or_invalid_amount"] = txns["amount_status"] != "Valid"

# Summary & separation of operational vs. financial views
print(txns["amount_status"].value_counts())
print("\nFlag distribution:")
print(txns["is_missing_or_invalid_amount"].value_counts())

valid_financial_df = txns.loc[~txns["is_missing_or_invalid_amount"]].copy()
print(f"\nRetained for operational counts: {len(txns)} rows")
print(f"Usable for financial sums:      {len(valid_financial_df)} rows")

amount_status
Valid              580
Missing (Blank)     15
Invalid ('N/A')      5
Name: count, dtype: int64

Flag distribution:
is_missing_or_invalid_amount
False    580
True      20
Name: count, dtype: int64

Retained for operational counts: 600 rows
Usable for financial sums:      580 rows


In [18]:
import pandas as pd

# Load datasets and apply prior deduplication & type conversions
agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

valid_agents = set(agents["agent_id"])
txns["_valid_agent"] = txns["agent_id"].isin(valid_agents)
txns = (
    txns.drop_duplicates()
        .sort_values(["txn_id", "_valid_agent"], ascending=[True, False])
        .drop_duplicates(subset=["txn_id"], keep="first")
        .drop(columns=["_valid_agent"])
        .reset_index(drop=True)
)
txns["txn_type"] = txns["txn_type"].astype(str).str.strip().str.title()
txns["txn_date"] = pd.to_datetime(txns["txn_date"].astype(str).str.strip(), format="mixed")
txns["amount"] = pd.to_numeric(txns["amount"], errors="coerce")

# 1. Compute IQR bounds on non-null amounts
valid_amt = txns["amount"].dropna()
q1 = valid_amt.quantile(0.25)
q3 = valid_amt.quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
extreme_upper_fence = q3 + 3.0 * iqr

txns["is_iqr_outlier"] = (txns["amount"] < lower_fence) | (txns["amount"] > upper_fence)
txns["is_extreme_outlier"] = txns["amount"] > extreme_upper_fence

print(f"Q1: KES {q1:,.2f} | Median: KES {valid_amt.median():,.2f} | Q3: KES {q3:,.2f} | IQR: KES {iqr:,.2f}")
print(f"1.5*IQR Upper Fence: KES {upper_fence:,.2f} ({txns['is_iqr_outlier'].sum()} flagged)")
print(f"3.0*IQR Upper Fence: KES {extreme_upper_fence:,.2f} ({txns['is_extreme_outlier'].sum()} extreme)")

# 2. Investigate the KES 250,000 transactions & compare against each agent's normal activity
kes_250k = txns[txns["amount"] == 250000].merge(agents, on="agent_id", how="left")
print("\n--- KES 250,000 Transactions ---")
print(kes_250k[["txn_id", "agent_id", "county", "txn_type", "amount", "txn_date"]].to_string(index=False))

baselines = (
    txns[txns["agent_id"].isin(kes_250k["agent_id"]) & (txns["amount"] < 250000)]
    .groupby("agent_id")["amount"]
    .agg(other_txns="count", normal_median="median", normal_max="max")
)
print("\n--- Normal Baseline for Affected Agents (Excluding KES 250,000) ---")
print(baselines.to_string())

Q1: KES 1,077.00 | Median: KES 1,776.50 | Q3: KES 3,080.00 | IQR: KES 2,003.00
1.5*IQR Upper Fence: KES 6,084.50 (40 flagged)
3.0*IQR Upper Fence: KES 9,089.00 (13 extreme)

--- KES 250,000 Transactions ---
 txn_id agent_id  county   txn_type   amount   txn_date
TX00307    AG012  Kisumu    Deposit 250000.0 2026-03-02
TX00316    AG014 Nairobi    Deposit 250000.0 2026-04-01
TX00425    AG005 Mombasa Withdrawal 250000.0 2026-03-02
TX00464    AG009  Nakuru Withdrawal 250000.0 2026-04-11
TX00595    AG016  Kiambu    Deposit 250000.0 2026-03-11

--- Normal Baseline for Affected Agents (Excluding KES 250,000) ---
          other_txns  normal_median  normal_max
agent_id                                       
AG005             27         1568.0      5897.0
AG009             23         1301.0      7946.0
AG012             25         1659.0      7146.0
AG014             29         1992.0     13362.0
AG016             32         2038.0      8518.0


In [19]:
import pandas as pd

# Load master agent register and raw transaction log
agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

# Apply prior deduplication (preferring valid agent_ids on duplicate txn_id) & type cleaning
valid_agents = set(agents["agent_id"])
txns["_valid_agent"] = txns["agent_id"].isin(valid_agents)
txns = (
    txns.drop_duplicates()
        .sort_values(["txn_id", "_valid_agent"], ascending=[True, False])
        .drop_duplicates(subset=["txn_id"], keep="first")
        .drop(columns=["_valid_agent"])
        .reset_index(drop=True)
)
txns["txn_type"] = txns["txn_type"].astype(str).str.strip().str.title()
txns["txn_date"] = pd.to_datetime(txns["txn_date"].astype(str).str.strip(), format="mixed")
txns["amount"] = pd.to_numeric(txns["amount"], errors="coerce")
agents["onboarded"] = pd.to_datetime(agents["onboarded"])

# 1. Perform a left join with indicator=True to audit every transaction against agents.csv
merged_audit = txns.merge(agents, on="agent_id", how="left", indicator=True)

# 2. Isolate unknown-agent transactions for compliance/onboarding audit
unknown_txns = merged_audit.loc[merged_audit["_merge"] == "left_only"].copy()
print(f"Transactions referencing unknown agents (post-dedup): {len(unknown_txns)}")
print(f"Unknown agent IDs found: {unknown_txns['agent_id'].unique().tolist()}")
print(f"Total KES volume in unknown-agent rows: KES {unknown_txns['amount'].sum():,.2f}\n")
print(unknown_txns[["txn_id", "agent_id", "txn_type", "amount", "txn_date"]].to_string(index=False))

# 3. Filter to matched rows (`_merge == "both"`) for clean county & agent analytics
clean_merged = (
    merged_audit.loc[merged_audit["_merge"] == "both"]
                .drop(columns=["_merge"])
                .reset_index(drop=True)
)
print(f"\nVerified registered-agent transactions: {len(clean_merged)} rows")
print("\nVerified transaction count by county:")
print(clean_merged["county"].value_counts())

Transactions referencing unknown agents (post-dedup): 7
Unknown agent IDs found: ['AG099']
Total KES volume in unknown-agent rows: KES 27,156.00

 txn_id agent_id   txn_type  amount   txn_date
TX00056    AG099    Deposit  8507.0 2026-04-01
TX00069    AG099 Withdrawal  1877.0 2026-04-24
TX00078    AG099    Deposit  1770.0 2026-04-06
TX00238    AG099    Deposit  4944.0 2026-04-06
TX00268    AG099    Deposit  3110.0 2026-04-09
TX00445    AG099    Deposit  5309.0 2026-04-27
TX00558    AG099    Deposit  1639.0 2026-03-22

Verified registered-agent transactions: 593 rows

Verified transaction count by county:
county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Name: count, dtype: int64


In [21]:
import pandas as pd
from generate_data import generate_all
from cleaning_pipeline.csv import clean_agent_data

# 1. Regenerate all CSVs reproducibly
generate_all()

# 2. Load raw agent register and transaction log, then run cleaning pipeline
agents_df = pd.read_csv("agents.csv")
txns_df = pd.read_csv("agent_transactions.csv")
clean_df, log = clean_agent_data(txns_df, agents_df)

print("=== CLEANING AUDIT LOG ===")
for k, v in log.items():
    print(f"{k}: {v}")

# 3. (a) County-level summary split by transaction type
county_summary = (
    clean_df.groupby(["county", "txn_type"], as_index=False)
    .agg(
        total_value=("amount", "sum"),
        txn_count=("txn_id", "count"),
        avg_value=("amount", "mean"),
    )
    .sort_values(["county", "txn_type"])
)
county_summary["avg_value"] = county_summary["avg_value"].round(2)

# 4. (b) Weekly transaction value pivot table by county
clean_df["week_start"] = (
    clean_df["txn_date"].dt.to_period("W").apply(lambda p: p.start_time.strftime("%Y-%m-%d"))
)
county_order = (
    clean_df.groupby("county")["amount"].sum().sort_values(ascending=False).index.tolist()
)
weekly_pivot = pd.pivot_table(
    clean_df,
    values="amount",
    index="week_start",
    columns="county",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total",
)[county_order + ["Total"]]

ModuleNotFoundError: No module named 'cleaning_pipeline'

In [22]:
"""Reusable data-cleaning pipeline module for the Mobile Money Agent Network."""

from __future__ import annotations

import numpy as np
import pandas as pd


def clean_agent_data(
    txns: pd.DataFrame,
    agents: pd.DataFrame,
    drop_missing_amounts: bool = True,
    drop_extreme_outliers: bool = True,
) -> tuple[pd.DataFrame, dict]:
    """Clean agent transactions and merge with the registered agent register.

    Applies deterministic, rule-based cleaning steps without hard-coded row indices:
      1. **Uniqueness:** Drops exact duplicate rows and resolves duplicate `txn_id`s
         by preferring records linked to a registered `agent_id`.
      2. **Consistency:** Strips whitespace and standardizes `txn_type` into strictly
         `"Deposit"` and `"Withdrawal"`, guarding against `NaN` and `"N/A"` strings.
      3. **Validity & Types:** Strips formatting artifacts and coerces `amount` to
         `float64`, and parses mixed date formats (`YYYY-MM-DD`, `YYYY/MM/DD`) into
         `datetime64[ns]`.
      4. **Completeness:** Flags missing or non-positive amounts (`is_missing_or_invalid_amount`)
         without imputing synthetic values, and optionally filters them out.
      5. **Outliers (IQR):** Flags standard ($Q_3 + 1.5 \\times \\text{IQR}$) and extreme
         ($Q_3 + 3.0 \\times \\text{IQR}$) upper outliers, isolating injected KES 250,000
         anomalies while preserving legitimate right-skewed transactions.
      6. **Referential Integrity (Merge):** Audits transactions against `agents['agent_id']`,
         logs unknown/orphan agent IDs (e.g., `AG099`), and joins `county` and `onboarded`.

    Args:
        txns: Raw agent transactions DataFrame.
        agents: Registered agent master DataFrame (`agent_id`, `county`, `onboarded`).
        drop_missing_amounts: If True (default), exclude rows with missing/non-positive
            amounts from the returned clean DataFrame after flagging and logging them.
        drop_extreme_outliers: If True (default), exclude extreme $3.0 \\times \\text{IQR}$
            outliers (such as the KES 250,000 anomalies) after flagging and logging them.

    Returns:
        A tuple `(clean_df, log)` where `clean_df` is the cleaned, enriched DataFrame
        and `log` is a structured audit dictionary of counts and anomalies at each step.
    """
    log: dict = {
        "initial_txn_rows": int(len(txns)),
        "initial_agent_rows": int(len(agents)),
    }

    df = txns.copy()
    agents_clean = agents.copy()

    # Standardize keys & agent register metadata
    agents_clean["agent_id"] = agents_clean["agent_id"].astype(str).str.strip().str.upper()
    agents_clean["county"] = agents_clean["county"].astype(str).str.strip().str.title()
    agents_clean["onboarded"] = pd.to_datetime(agents_clean["onboarded"], format="mixed")
    valid_agents = set(agents_clean["agent_id"])

    df["txn_id"] = df["txn_id"].astype(str).str.strip().str.upper()
    df["agent_id"] = df["agent_id"].astype(str).str.strip().str.upper()

    # 1. UNIQUENESS: Remove exact duplicates, then deduplicate on txn_id preferring valid agents
    exact_dups = int(df.duplicated(keep="first").sum())
    df = df.drop_duplicates(keep="first").copy()
    log["exact_duplicates_removed"] = exact_dups

    df["_valid_agent"] = df["agent_id"].isin(valid_agents)
    df = df.sort_values(by=["txn_id", "_valid_agent"], ascending=[True, False])
    id_dups = int(df.duplicated(subset=["txn_id"], keep="first").sum())
    df = (
        df.drop_duplicates(subset=["txn_id"], keep="first")
          .drop(columns=["_valid_agent"])
          .reset_index(drop=True)
    )
    log["duplicate_txn_ids_removed"] = id_dups
    log["total_duplicates_removed"] = exact_dups + id_dups

    # 2. CONSISTENCY: Standardize txn_type into {"Deposit", "Withdrawal"}
    invalid_tokens = {"", "N/A", "NA", "NONE", "NULL", "NAN"}
    raw_type_str = df["txn_type"].astype(str).str.strip()
    invalid_type_mask = df["txn_type"].isna() | raw_type_str.str.upper().isin(invalid_tokens)

    standardized_type = np.where(
        invalid_type_mask,
        np.nan,
        raw_type_str.str.title(),
    )
    log["txn_type_values_standardized"] = int(
        (df["txn_type"].fillna("").astype(str) != pd.Series(standardized_type).fillna("")).sum()
    )
    df["txn_type"] = standardized_type

    allowed_types = {"Deposit", "Withdrawal"}
    unexpected_types = set(df["txn_type"].dropna()) - allowed_types
    if unexpected_types:
        raise ValueError(f"Unexpected txn_type values encountered: {unexpected_types}")
    log["unique_txn_types"] = sorted(df["txn_type"].dropna().unique().tolist())

    # 3. VALIDITY & TYPES: Coerce amount to numeric and txn_date to datetime64[ns]
    raw_amount_str = df["amount"].astype(str).str.strip()
    cleaned_amount_str = (
        raw_amount_str
        .str.replace(r"(?i)^kes\s*", "", regex=True)
        .str.replace(",", "", regex=False)
    )
    df["amount"] = pd.to_numeric(cleaned_amount_str, errors="coerce")

    raw_date_str = df["txn_date"].astype(str).str.strip()
    log["slash_dates_normalized"] = int(raw_date_str.str.contains("/", regex=False).sum())
    df["txn_date"] = pd.to_datetime(raw_date_str, format="mixed", errors="coerce")
    invalid_dates = int(df["txn_date"].isna().sum())
    log["invalid_dates_dropped"] = invalid_dates
    df = df.loc[df["txn_date"].notna()].copy()

    # 4. COMPLETENESS: Flag missing/invalid amounts (no imputation)
    missing_mask = df["amount"].isna() | (df["amount"] <= 0)
    df["is_missing_or_invalid_amount"] = missing_mask
    log["missing_or_invalid_amounts_flagged"] = int(missing_mask.sum())

    # 5. OUTLIERS (IQR): Compute fences on valid numeric amounts & flag anomalies
    valid_amt = df.loc[~missing_mask, "amount"]
    q1 = float(valid_amt.quantile(0.25))
    q3 = float(valid_amt.quantile(0.75))
    iqr = q3 - q1
    upper_fence_15 = q3 + 1.5 * iqr
    upper_fence_30 = q3 + 3.0 * iqr

    df["is_iqr_outlier"] = (~missing_mask) & (df["amount"] > upper_fence_15)
    df["is_extreme_outlier"] = (~missing_mask) & (df["amount"] > upper_fence_30)

    log["iqr_q1"] = round(q1, 2)
    log["iqr_q3"] = round(q3, 2)
    log["iqr_upper_fence_1_5x"] = round(upper_fence_15, 2)
    log["iqr_upper_fence_3_0x"] = round(upper_fence_30, 2)
    log["iqr_outliers_flagged_1_5x"] = int(df["is_iqr_outlier"].sum())
    log["extreme_outliers_flagged_3_0x"] = int(df["is_extreme_outlier"].sum())
    log["kes_250k_anomalies_flagged"] = int((df["amount"] == 250000).sum())

    if drop_missing_amounts:
        df = df.loc[~df["is_missing_or_invalid_amount"]].copy()
    if drop_extreme_outliers:
        df = df.loc[~df["is_extreme_outlier"]].copy()

    # 6. MERGE & REFERENTIAL INTEGRITY: Join to registered agents and isolate orphans
    merged = df.merge(agents_clean, on="agent_id", how="left", indicator=True)
    orphan_mask = merged["_merge"] == "left_only"
    log["unknown_agent_ids"] = sorted(merged.loc[orphan_mask, "agent_id"].unique().tolist())
    log["unknown_agent_rows_dropped"] = int(orphan_mask.sum())

    clean_df = (
        merged.loc[merged["_merge"] == "both"]
              .drop(columns=["_merge"])
              .sort_values("txn_id")
              .reset_index(drop=True)
    )
    log["final_clean_rows"] = int(len(clean_df))
    log["total_rows_removed"] = int(log["initial_txn_rows"] - log["final_clean_rows"])

    return clean_df, log

Data Quality Report

### Data Privacy & Protection Under the Kenya Data Protection Act, 2019

If the transaction extract also included customer phone numbers, **`customer_phone`** would constitute direct **personal data** under Section 2 of the **Kenya Data Protection Act, 2019 (KDPA)** (as a unique identifier linked to a natural person), while **`agent_id`** (identifying individual registered agents, especially when paired with `county` and `onboarded` date) and transaction-level metadata (**`txn_id`**, **`txn_date`**, **`amount`**, **`county`**) become **indirectly identifiable personal data** whenever they can be linked back to an identifiable individual. To comply with KDPA principles of **data minimization** (Section 25(c)), **purpose limitation** (Section 25(b)), and **storage limitation / integrity and confidentiality** (Section 25(e)–(f) and Section 41), analytics workflows should strip raw phone numbers before analysis, replace them with a **keyed cryptographic pseudonym (`HMAC-SHA256`)** using a secret salt stored outside the codebase/repository (allowing repeat-customer cohorting without exposing MSISDNs), enforce role-based access controls and encryption at rest/in transit, and apply defined retention schedules so raw identifiers are never persisted in exploratory CSVs or notebooks.

In [12]:
import hashlib
import hmac
import os
import pandas as pd


def protect_personal_data(
    df: pd.DataFrame,
    phone_col: str = "customer_phone",
    salt_env_var: str = "DPA_PSEUDONYM_SALT",
) -> pd.DataFrame:
    """Protect personal data under the Kenya Data Protection Act 2019.

    Applies data minimization and keyed pseudonymization:
      1. Normalizes Kenyan phone formats (e.g., '07...' -> '+2547...') so the same
         customer maps deterministically across transactions.
      2. Replaces raw MSISDNs with an HMAC-SHA256 token keyed by an external secret
         salt (never hard-coded in notebooks or version control).
      3. Drops the raw `customer_phone` column from the analytical DataFrame.
    """
    if phone_col not in df.columns:
        return df.copy()

    secret_salt = os.environ.get(salt_env_var)
    if not secret_salt:
        raise EnvironmentError(
            f"Missing required secret environment variable '{salt_env_var}' for KDPA pseudonymization."
        )

    protected = df.copy()

    # 1. Normalize Kenyan phone numbers to E.164 (+254XXXXXXXXX)
    normalized_phone = (
        protected[phone_col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
        .str.replace(r"^0", "+254", regex=True)
    )

    # 2. Keyed HMAC-SHA256 pseudonymization (prevents rainbow-table reversal)
    key_bytes = secret_salt.encode("utf-8")
    protected["customer_token"] = normalized_phone.apply(
        lambda p: hmac.new(key_bytes, p.encode("utf-8"), hashlib.sha256).hexdigest()[:16]
    )

    # 3. Data minimization: remove raw phone numbers from analytical extract
    protected = protected.drop(columns=[phone_col])

    return protected


# Example usage:
os.environ["DPA_PSEUDONYM_SALT"] = "replace-with-kms-or-env-secret"
sample_df = pd.DataFrame({
    "txn_id": ["TX00001", "TX00002", "TX00004"],
    "agent_id": ["AG009", "AG020", "AG009"],
    "customer_phone": ["+254712345678", "0722987654", "+254733112233"],
    "txn_type": ["Deposit", "Deposit", "Deposit"],
    "amount": [353.0, 1238.0, 652.0],
    "county": ["Nakuru", "Mombasa", "Nakuru"],
})

safe_df = protect_personal_data(sample_df)
print(safe_df.to_string(index=False))

 txn_id agent_id txn_type  amount  county   customer_token
TX00001    AG009  Deposit   353.0  Nakuru 903c3bb04e4d08d7
TX00002    AG020  Deposit  1238.0 Mombasa 0bce9b8f5cc68133
TX00004    AG009  Deposit   652.0  Nakuru 41f8dff63a39e2e5
